# Задание 4

## Часть 1 — Сборка с Velvet

---

1. Взял файлы, которые мы использовали на семинаре (`7_S4_L001_R1_001.fastq`, `7_S4_L001_R2_001.fastq`)

2. Сделал несколько сборок с разными кмерами (6, 11, 21, 31)  с помощью программы `Velvet`. Выбрал именно такие числа, так как если запускать с k-mer > 31, выходит ошибка: 

```bash
cat velvet_66707.err 
velveth: Word length 33 greater than max allowed value (31).
```

---

### Скрипт - velvet.slurm

```bash
#!/bin/bash
#SBATCH --job-name=velvet
#SBATCH --cpus-per-task=8
#SBATCH --mem=1gb
#SBATCH --time=11:00:00
#SBATCH --output=velvet_%j.out
#SBATCH --error=velvet_%j.err
#SBATCH --partition=AMD9554

VELVET=/home/STUDY/FBMF/bioinformatics/soft/velvet
READ1=~/seminars/seminar_11/7_S4_L001_R1_001.fastq
READ2=~/seminars/seminar_11/7_S4_L001_R2_001.fastq
OUTDIR=~/seminars/sem11/genome_assembly_results/velvet

for K in 6 11 21 31
do
    echo "Running Velvet with k=$K"

    $VELVET/velveth $OUTDIR/k$K $K -fastq -shortPaired $READ1 $READ2

    $VELVET/velvetg $OUTDIR/k$K -ins_length 300
done
```

**Пояснение:** Когда я обращался к файлам в директориях (например: /home/STUDY/FBMF/bioinformatics/genome_de_novo/7_S4_L001_R1_001.fastq) у меня был Permission denied. Возможно, это можно было пофиксить, но я на семинаре скопировал себе в `seminar_11` эти файлы и использовал их.

---

### Результаты работы скрипта

```bash 
[studfbmf02_06@calc k6]$ ls -la --time=ctime ~/seminars/sem11/genome_assembly_results/velvet/
total 0
drwxr-xr-x. 6 studfbmf02_06 fbmf 4 Apr 26 14:02 .
drwxr-xr-x. 4 studfbmf02_06 fbmf 2 Apr 26 13:46 ..
drwxr-xr-x. 2 studfbmf02_06 fbmf 8 Apr 26 14:00 k11
drwxr-xr-x. 2 studfbmf02_06 fbmf 8 Apr 26 14:00 k21
drwxr-xr-x. 2 studfbmf02_06 fbmf 8 Apr 26 14:00 k31
drwxr-xr-x. 2 studfbmf02_06 fbmf 8 Apr 26 14:00 k6
[studfbmf02_06@calc k6]$ ls -la --time=ctime ~/seminars/sem11/genome_assembly_results/velvet/k31/
total 23779
drwxr-xr-x. 2 studfbmf02_06 fbmf        8 Apr 26 14:00 .
drwxr-xr-x. 6 studfbmf02_06 fbmf        4 Apr 26 14:02 ..
-rw-r--r--. 1 studfbmf02_06 fbmf    71714 Apr 26 14:00 contigs.fa
-rw-r--r--. 1 studfbmf02_06 fbmf   235448 Apr 26 14:00 Graph
-rw-r--r--. 1 studfbmf02_06 fbmf   235448 Apr 26 14:00 LastGraph
-rw-r--r--. 1 studfbmf02_06 fbmf     3351 Apr 26 14:00 Log
-rw-r--r--. 1 studfbmf02_06 fbmf   718882 Apr 26 14:00 PreGraph
-rw-r--r--. 1 studfbmf02_06 fbmf  6501882 Apr 26 14:00 Roadmaps
-rw-r--r--. 1 studfbmf02_06 fbmf 16404727 Apr 26 14:00 Sequences
-rw-r--r--. 1 studfbmf02_06 fbmf   176344 Apr 26 14:00 stats.txt
```

Скриншот

![image](pictures/part1_bioinf.png)

**Микро-вывод:** Было выполнено тестирование сборки Velvet при различных значениях k-mer (6, 11, 21, 31). При малых значениях k наблюдается высокая фрагментация сборки. Увеличение k приводит к уменьшению числа контигов и увеличению их длины. Наилучшие результаты среди Velvet были получены при k=31

## Часть 2 — Сравнение сборок

---

1. С помощью `QUAST` сравнил сборки `Velvet` и `SPAdes` командой:

```bash
 /home/STUDY/FBMF/bioinformatics/soft/bin/quast.py  -o assembly_analysis/report  -m 0  --threads 1  genome_assembly_results/spades/scaffolds.fasta  genome_assembly_results/velvet/k31/contigs.fa
```

#### Таблица

![image](pictures/part_2_quast.png)

2. **Вывод:** 

Сборка, выполненная с использованием SPAdes, показала более высокое качество по сравнению с Velvet. Это выражается в меньшем количестве контигов, более высоком значении N50 и меньшей фрагментации сборки.

Velvet же сформировал значительно более фрагментированную сборку с большим числом коротких контигов и низкими значениями N50 и L50.

SPAdes обеспечивает более качественную реконструкцию генома за счёт более эффективной обработки ошибок чтений и лучшего использования информации paired-end



## Часть 3 — Улучшение сборки

--- 

#### Улучшение SPAdes

```bash
python3 /home/STUDY/FBMF/bioinformatics/soft/SPAdes-4.2.0-Linux/bin/spades.py  --careful -1 ~/seminars/seminar_11/7_S4_L001_R1_001.fastq -2 ~/seminars/seminar_11/7_S4_L001_R2_001.fastq -k 21,33,55,77  -o ~/seminars/seminar_11/genome_assembly_results/spades_improved
```

Добавил multi-k сборку. Она должна улучшить метрики, так как 

- разные k-mer $\to$ лучше собирает повторы
- маленькие k $\to$ соединяют граф
- большие k $\to$ уменьшают ошибки
- combined $\to$ лучше N50 и меньше contigs

#### Улучшение Velvet

Изменённый скрипт

```bash
#!/bin/bash
#SBATCH --job-name=velvet_improved
#SBATCH --cpus-per-task=8
#SBATCH --mem=8gb
#SBATCH --time=12:00:00
#SBATCH --output=velvet_improved_%j.out
#SBATCH --error=velvet_improved_%j.err
#SBATCH --partition=AMD9554

VELVET=/home/STUDY/FBMF/bioinformatics/soft/velvet
READ1=~/seminars/seminar_11/7_S4_L001_R1_001.fastq
READ2=~/seminars/seminar_11/7_S4_L001_R2_001.fastq
OUTDIR=~/seminars/sem11/genome_assembly_results/velvet_improved

mkdir -p $OUTDIR

for K in 21 29 31 33
do
    echo "Running improved Velvet with k=$K"

    $VELVET/velveth $OUTDIR/k$K $K -fastq -shortPaired $READ1 $READ2

    $VELVET/velvetg $OUTDIR/k$K \
        -ins_length 300 \
        -exp_cov auto \
        -cov_cutoff auto
done
```

Изменения:
- добавлены средние k $\to$ лучше баланс графа
- добавлены параметры velvetg, которые убирают ошибки секвенирования, удаляют низкокачественные k-mers, улучшают связность графа:

```bash
-exp_cov auto
-cov_cutoff auto
```


#### Сравнение 4 сборок 

```bash 
 /home/STUDY/FBMF/bioinformatics/soft/bin/quast.py \
 -o assembly_analysis/report_final \
 -m 0 \
 --threads 1 \
 genome_assembly_results/spades/scaffolds.fasta \
 genome_assembly_results/spades_improved/contigs.fasta \
 genome_assembly_results/velvet/k31/contigs.fa \
 genome_assembly_results/velvet_improved/k31/contigs.fa
```

![image](pictures/improved.png)

**Вывод по улучшениям:** Сравнение четырёх сборок с помощью QUAST показало, что улучшенные версии SPAdes и Velvet не изменили метрики по сравнению с исходными сборками ( 

Для SPAdes обе версии имеют одинаковые показатели: 49 контигов, N50 = 440 и L50 = 12

Для Velvet улучшенная и исходная сборки также идентичны по метрикам


## Вывод

---

В ходе работы были выполнены сборки генома с использованием двух инструментов: `SPAdes` и `Velvet` - при различных параметрах k-mer. В первой части были получены базовые сборки с фиксированными параметрами, во второй части проведено их сравнение с использованием `QUAST`.

Анализ показал, что `SPAdes` обеспечивает более качественную сборку по сравнению с `Velvet`: характеризуется меньшим количеством контигов, более высоким значением N50 и меньшей фрагментацией сборки.

В третьей части были предприняты попытки улучшения качества сборок путём оптимизации параметров (изменение k-mer и использование режимов улучшенной сборки). Однако значительного улучшения метрик по результатам `QUAST` не наблюдалось, и SPAdes остался более стабильным и качественным методом сборки по сравнению с `Velvet`